# **Baseline Models**
**Decodelabs Internship | Week 2 | Task 5 (Part 1)**

---
Here, I trained two baseline models: a Dummy Classifier and a
Logistic Regression.


In [1]:
import sys, os
NOTEBOOK_DIR = os.getcwd()
PROJECT_ROOT = os.path.dirname(NOTEBOOK_DIR)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
from configs.config import (
    RAW_FILE, IDS_MAP_FILE, INTERIM_FILE, PROCESSED_FILE,
    TRAIN_FILE, VAL_FILE, TEST_FILE,
    FIGURES_DIR, TABLES_DIR, PAPER_FIG_DIR, PAPER_TAB_DIR,
    RANDOM_SEED, TARGET_COL, PATIENT_ID_COL, MEDICATION_COLS,
    AGE_ORDER, icd9_to_category, COLORS, ensure_dirs
)
from src.plot_utils import set_plot_style, save_figure, save_table
ensure_dirs()
set_plot_style()
print("Config loaded. Seed:", RANDOM_SEED)


Config loaded. Seed: 42


In [ ]:
import pandas as pd
import numpy as np
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings("ignore")

train_df = pd.read_csv(TRAIN_FILE)
val_df   = pd.read_csv(VAL_FILE)

X_train = train_df.drop(columns=[TARGET_COL])
y_train = train_df[TARGET_COL]
X_val   = val_df.drop(columns=[TARGET_COL])
y_val   = val_df[TARGET_COL]

print(f"Training set  : {len(X_train):,} rows | {y_train.mean()*100:.1f}% positive")
print(f"Validation set: {len(X_val):,} rows  | {y_val.mean()*100:.1f}% positive")
print(f"Features      : {X_train.shape[1]}")

Training set  : 48,990 rows | 7.5% positive
Validation set: 10,498 rows  | 7.5% positive
Features      : 83


## Dummy Classifier (absolute baseline)

The Dummy Classifier predicts the most frequent class (0) every single time. This is the baseline: any real model must beat this. On an imbalanced dataset (~89% class 0), this gives ~89% accuracy but 0% recall for class 1 — completely useless clinically.

In [3]:
dummy = DummyClassifier(strategy="most_frequent", random_state=RANDOM_SEED)
dummy.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_val)

print("=== Dummy Classifier on Validation Set ===")
print(classification_report(y_val, y_pred_dummy,
      target_names=["No Readmit (0)", "Early Readmit (1)"]))
print()
print("As expected: Recall for class 1 = 0.00")
print("The dummy model catches NONE of the early readmissions.")
print("Any model I build must have class-1 Recall > 0.")

=== Dummy Classifier on Validation Set ===
                   precision    recall  f1-score   support

   No Readmit (0)       0.93      1.00      0.96      9712
Early Readmit (1)       0.00      0.00      0.00       786

         accuracy                           0.93     10498
        macro avg       0.46      0.50      0.48     10498
     weighted avg       0.86      0.93      0.89     10498


As expected: Recall for class 1 = 0.00
The dummy model catches NONE of the early readmissions.
Any model I build must have class-1 Recall > 0.


## Logistic Regression with balanced class weight

Logistic Regression is a linear model that estimates the probability of class 1. `class_weight='balanced'` automatically adjusts weights inversely proportional to class frequency, which helps the model pay more attention to the rare class (1).

In [4]:
logreg = LogisticRegression(
    class_weight="balanced",   # handles class imbalance
    max_iter=2000,             # allow enough iterations to converge
    random_state=RANDOM_SEED,
    solver="lbfgs",            # efficient solver for medium-sized datasets
    C=1.0                      # regularisation strength (1.0 = default)
)

print("Training Logistic Regression...")
logreg.fit(X_train, y_train)
print("Training complete.")

Training Logistic Regression...
Training complete.


In [5]:
# Evaluate on validation set
y_pred_lr = logreg.predict(X_val)
y_prob_lr = logreg.predict_proba(X_val)[:, 1]

from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

print("=== Logistic Regression on Validation Set ===")
print(classification_report(y_val, y_pred_lr,
      target_names=["No Readmit (0)", "Early Readmit (1)"]))

roc_auc = roc_auc_score(y_val, y_prob_lr)
avg_prec = average_precision_score(y_val, y_prob_lr)
print(f"ROC-AUC          : {roc_auc:.4f}")
print(f"Average Precision: {avg_prec:.4f}")
print()
print("Interpretation:")
print(f"  ROC-AUC {roc_auc:.3f} means the model ranks a random positive higher")
print(f"  than a random negative {roc_auc*100:.1f}% of the time.")

=== Logistic Regression on Validation Set ===
                   precision    recall  f1-score   support

   No Readmit (0)       0.95      0.67      0.79      9712
Early Readmit (1)       0.12      0.56      0.20       786

         accuracy                           0.66     10498
        macro avg       0.54      0.62      0.49     10498
     weighted avg       0.89      0.66      0.74     10498

ROC-AUC          : 0.6537
Average Precision: 0.1425

Interpretation:
  ROC-AUC 0.654 means the model ranks a random positive higher
  than a random negative 65.4% of the time.


## Cross-validation on training data

I ran 5-fold stratified cross-validation on the training data. This gives a more reliable estimate of performance than a single val split. `StratifiedKFold` preserves class proportions in each fold.

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

cv_results = cross_validate(
    logreg, X_train, y_train,
    cv=cv,
    scoring=["roc_auc", "f1", "recall", "precision"],
    return_train_score=True
)

print("=== 5-Fold Cross-Validation: Logistic Regression ===")
for metric in ["roc_auc", "f1", "recall", "precision"]:
    test_scores = cv_results[f"test_{metric}"]
    train_scores = cv_results[f"train_{metric}"]
    print(f"  {metric:12s}  Train: {train_scores.mean():.3f}±{train_scores.std():.3f}  "
          f"Val: {test_scores.mean():.3f}±{test_scores.std():.3f}")

print()
print("If Train >> Val scores, the model is overfitting.")
print("If Train ≈ Val, generalisation is good.")

=== 5-Fold Cross-Validation: Logistic Regression ===
  roc_auc       Train: 0.672±0.002  Val: 0.661±0.007
  f1            Train: 0.203±0.002  Val: 0.198±0.004
  recall        Train: 0.573±0.003  Val: 0.558±0.015
  precision     Train: 0.123±0.001  Val: 0.120±0.002

If Train >> Val scores, the model is overfitting.
If Train ≈ Val, generalisation is good.


## Inspecting Logistic Regression coefficients

For Logistic Regression, the coefficients show which features push the predicted probability up (positive) or down (negative).

In [7]:
coefs = pd.Series(logreg.coef_[0], index=X_train.columns).sort_values(key=abs, ascending=False)
top_positive = coefs[coefs > 0].head(10)
top_negative = coefs[coefs < 0].head(10)

print("=== Top 10 Positive Coefficients (increase readmission probability) ===")
print(top_positive.round(4).to_string())
print()
print("=== Top 10 Negative Coefficients (decrease readmission probability) ===")
print(top_negative.round(4).to_string())

=== Top 10 Positive Coefficients (increase readmission probability) ===
number_inpatient            0.2692
total_prior_visits          0.2408
discharge_disposition_id    0.1685
age_ordinal                 0.1528
race_Caucasian              0.1141
num_lab_procedures          0.1087
on_diabetes_med             0.1004
glucose_tested              0.0836
time_in_hospital            0.0654
diag_1_cat_Circulatory      0.0646

=== Top 10 Negative Coefficients (decrease readmission probability) ===
number_outpatient        -0.1103
tolazamide               -0.1083
max_glu_serum_encoded    -0.0701
diag_1_cat_Respiratory   -0.0684
med_changed              -0.0660
had_prior_inpatient      -0.0643
glipizide-metformin      -0.0588
n_medication_increases   -0.0561
diag_2_cat_Diabetes      -0.0466
diag_2_cat_Respiratory   -0.0416


## Baseline metrics saved

In [8]:
from src.eval_utils import compute_metrics

baseline_results = [
    compute_metrics(y_val, dummy.predict(X_val), None, "Dummy Classifier"),
    compute_metrics(y_val, y_pred_lr, y_prob_lr, "Logistic Regression"),
]

import json
results_path = os.path.join(TABLES_DIR, "07_baseline_results.json")
os.makedirs(TABLES_DIR, exist_ok=True)
with open(results_path, "w") as f:
    json.dump(baseline_results, f, indent=2)
print(f"Baseline results saved: {results_path}")

for r in baseline_results:
    print(f"\n  {r['Model']}")
    for k, v in r.items():
        if k != "Model":
            print(f"    {k:20s}: {v}")

Baseline results saved: c:\Users\Peter\Documents\EXTRA-CURRICULA\Internship\Decodelab\diabetes_readmission\reports\tables\07_baseline_results.json

  Dummy Classifier
    Accuracy            : 0.9251
    Balanced Acc.       : 0.5
    Precision           : 0.0
    Recall              : 0.0
    F1-Score            : 0.0
    ROC-AUC             : N/A
    Avg. Precision      : N/A
    TP                  : 0
    TN                  : 9712
    FP                  : 0
    FN                  : 786

  Logistic Regression
    Accuracy            : 0.6611
    Balanced Acc.       : 0.6151
    Precision           : 0.1207
    Recall              : 0.5611
    F1-Score            : 0.1986
    ROC-AUC             : 0.6537
    Avg. Precision      : 0.1425
    TP                  : 441
    TN                  : 6499
    FP                  : 3213
    FN                  : 345
